In [4]:
import cv2
from ultralytics import YOLO
import pygame  # To play the sound without Arduino

# 1. Setup Sound
pygame.mixer.init()
repellent_sound = pygame.mixer.Sound("mixkit-classic-alarm-995.wav") # Put your sound file name here

# 2. Load the AI
model = YOLO('yolov8n.pt') 
cap = cv2.VideoCapture(0)

# 3. Define the "Restricted Zone" line
ZONE_THRESHOLD = 350 

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    results = model(frame, conf=0.5)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2) # Center point

            # Draw the box and center point
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

            # TRIGGER LOGIC: If the center point crosses the line
            if cy > ZONE_THRESHOLD:
                cv2.putText(frame, "REPELLING ANIMAL...", (50, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                
                # Play the sound through laptop speakers
                if not pygame.mixer.get_busy(): # Don't overlap sounds
                    repellent_sound.play()

    # Draw the boundary line
    cv2.line(frame, (0, ZONE_THRESHOLD), (640, ZONE_THRESHOLD), (255, 255, 0), 2)
    cv2.imshow("Zero-Budget Wildlife Shield", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
pip install ultralytics opencv-python pygame